# Fase 2: Exploración, Limpieza Interanual, Transformación y Validación (`Base_Maestra_2023_2025.xlsx`)
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal
 
---
## 1. Configuración del Entorno e Importación del Módulo Reutilizable (`F2/src/preprocesamiento.py`)


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
 
# Asegurar que el directorio de trabajo sea la raíz del repositorio
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
 
from F2.src.preprocesamiento import (
    cargar_datos_seguro,
    diagnosticar_variables,
    estandarizar_regiones_y_peligrosidad,
    rescatar_coordenadas_chile,
    comparar_estrategias_nulos,
    imputar_por_grupo_con_flag,
)
 
sns.set_theme(style="whitegrid", palette="deep")
RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")
df_raw = cargar_datos_seguro(RUTA_RAW)
print(f"[OK] Dataset original cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")



## 2. Exploración Inicial, Diagnóstico de Calidad y Depuración Interanual (2023–2025)
Se inspecciona la estructura original del dataset (`118.280 filas × 21 columnas`), cuantificando:
1. **Registros duplicados exactos:** `118` filas duplicadas.
2. **Inconsistencias interanuales de formato:** `region` presenta `19` etiquetas para las 16 regiones de Chile y `peligrosidad` presenta `812` combinaciones por diferencias de mayúsculas, tildes y separadores (`##` en 2023–2024 vs `;` en 2025).
3. **Corrupción de escala decimal en coordenadas 2024:** En el año 2024 las coordenadas se exportaron sin separador decimal (ej. `-33447301481283` en lugar de `-33.447301`).


In [ ]:
# 2.1 Diagnóstico cuantitativo sobre el dataset crudo
tabla_diagnostico_raw = diagnosticar_variables(df_raw)
display(tabla_diagnostico_raw)
 
# 2.2 Eliminación de filas duplicadas exactas
n_duplicados = int(df_raw.duplicated().sum())
df_clean = df_raw.drop_duplicates().copy().reset_index(drop=True)
print(f"Duplicados exactos eliminados: {n_duplicados} -> Filas únicas: {len(df_clean)}")
 
# 2.3 Estandarización de regiones (19 -> 16) y peligrosidad (812 -> 93)
print(f"Antes de normalizar -> Regiones: {df_clean['region'].nunique()} | Peligrosidades: {df_clean['peligrosidad'].nunique()}")
df_clean = estandarizar_regiones_y_peligrosidad(df_clean)
print(f"Después de normalizar -> Regiones: {df_clean['region'].nunique()} | Peligrosidades: {df_clean['peligrosidad'].nunique()}")
 
# 2.4 Rescate de punto decimal y validación de límites geográficos de Chile en latitud y longitud
df_clean = rescatar_coordenadas_chile(df_clean)
tabla_diagnostico_pos = diagnosticar_variables(df_clean)
print("\nDiagnóstico de variables numéricas tras rescate de coordenadas:")
display(tabla_diagnostico_pos[tabla_diagnostico_pos["variable"].isin(["cantidad_kg", "latitud", "longitud"])])


## 3. Estrategia Fundamentada de Manejo de Datos Faltantes e Imputación con Trazabilidad
- **Descarte de `dropna()`:** Eliminar filas con valores faltantes descartaría **5.037 declaraciones (4,26% del total)**, omitiendo miles de toneladas de residuos peligrosos reportados al RETC.
- **Descarte de la media global en `cantidad_kg`:** La variable `cantidad_kg` presenta un **16,19% de valores atípicos extremos** (grandes complejos industriales y mineros con declaraciones de hasta 48.223 toneladas), lo que eleva la media a **16.422,33 kg** frente a una mediana de **234,00 kg** (70 veces menor). Imputar con la media inflaría artificialmente los registros faltantes.
- **Decisión adoptada:** Imputar las variables continuas (`cantidad_kg`, `latitud`, `longitud`) con la **mediana agrupada por `region`** y las categóricas (`id_lista_a`, `id_ciiu4`, `ciiu4`, `id_ciiu6`, `ciiu6`, `rubro`) con la **moda agrupada por `id_rubro`**, registrando columnas binarias `_imputado` (`0`/`1`).


In [ ]:
# 3.1 Tabla comparativa de estrategias de manejo de nulos sobre cantidad_kg y latitud
print("Comparación de estrategias sobre 'cantidad_kg':")
display(comparar_estrategias_nulos(df_clean, col_num="cantidad_kg", col_grupo="region"))
 
print("\nComparación de estrategias sobre 'latitud':")
display(comparar_estrategias_nulos(df_clean, col_num="latitud", col_grupo="region"))
 
# 3.2 Aplicar imputación por grupo con creación de columnas binarias '_imputado'
COLS_NUM_CON_NULOS = ["cantidad_kg", "latitud", "longitud"]
COLS_CAT_CON_NULOS = ["id_lista_a", "id_ciiu4", "ciiu4", "id_ciiu6", "ciiu6", "rubro"]
 
df_imputado = imputar_por_grupo_con_flag(
    df_clean,
    cols_numericas=COLS_NUM_CON_NULOS,
    cols_categoricas=COLS_CAT_CON_NULOS,
    col_grupo_num="region",
    col_grupo_cat="id_rubro",
)
 
print(f"\n[OK] Total de valores nulos tras la imputación: {int(df_imputado.isna().sum().sum())}")
 
# 3.3 Visualización: distribución log10(cantidad_kg) y coordenadas rescatadas e imputadas
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(np.log10(df_imputado["cantidad_kg"].clip(lower=1)), bins=40, kde=True, ax=axes[0], color="#1d4ed8")
axes[0].set_title("Distribución de log10(cantidad_kg) tras imputación")
axes[0].set_xlabel("log10(cantidad_kg)")
 
sns.boxplot(data=df_imputado, x="latitud", y="estado_materia", ax=axes[1], palette="Set2")
axes[1].set_title("Distribución geográfica (latitud) por estado físico del residuo")
plt.tight_layout()
plt.show()


## 4. Transformación y Codificación de Variables según su Rol Analítico
Se transforma cada familia de variables de acuerdo con su naturaleza estadística y regulatoria:
1. **Binarias (`0/1`):** `es_gran_generador_bin` ($\ge 12.000\text{ kg/año}$ según Art. 25 D.S. 148/2003) y `multi_peligrosidad_bin` (múltiples características de peligrosidad).
2. **Ordinal (`1..4`):** `estado_materia_ord` según movilidad física de contención (`sólido=1 < lodo=2 < líquido=3 < gaseoso=4`).
3. **Nominales (`region`, `estado_materia`):** Codificación *One-Hot Encoding* (`pd.get_dummies`).
4. **Alta cardinalidad (`comuna`, `rut_razon_social`):** Agrupación en Top-10 comunas + `'OTROS'` y codificación por frecuencia de declaración (`frecuencia_declaraciones_rut`).
5. **Temporal (`año`):** Índice de tendencia trianual (`anio_indice`) y codificación cíclica ($\sin/\cos$).
6. **Texto libre (`peligrosidad`, `ciiu4`):** Conteo de peligrosidades simultáneas (`n_peligrosidades`) y longitud léxica (`ciiu4_n_palabras`).


In [ ]:
df_trans = df_imputado.copy()
 
# 4.1 Variables Binarias (0 / 1) y de Persistencia de Establecimiento (id_vu entre 2023, 2024 y 2025)
anios_por_vu = df_trans.groupby("id_vu")["año"].transform("nunique")
df_trans["n_anios_declarados_vu"] = anios_por_vu.astype(int)
df_trans["es_generador_persistente_bin"] = (df_trans["n_anios_declarados_vu"] == 3).astype(int)
 
df_trans["n_peligrosidades"] = df_trans["peligrosidad"].apply(lambda x: len(str(x).split(" + ")))
df_trans["multi_peligrosidad_bin"] = (df_trans["n_peligrosidades"] > 1).astype(int)
df_trans["es_gran_generador_bin"] = (df_trans["cantidad_kg"] >= 12000.0).astype(int)
 
# 4.2 Variable Ordinal (escala física de movilidad/dispersión del residuo: 1 a 4)
ORDEN_ESTADO = {"sólido": 1, "lodo": 2, "líquido": 3, "gaseoso": 4}
df_trans["estado_materia_ord"] = df_trans["estado_materia"].map(ORDEN_ESTADO).fillna(1).astype(int)
 
# 4.3 Variables Temporales (índice trianual y codificación armónica)
df_trans["anio_indice"] = df_trans["año"] - 2023
df_trans["anio_sin"] = np.sin(2 * np.pi * df_trans["anio_indice"] / 3.0)
df_trans["anio_cos"] = np.cos(2 * np.pi * df_trans["anio_indice"] / 3.0)
 
# 4.4 Alta cardinalidad: Top-10 comunas + 'OTROS' y frecuencia por RUT declarante
top10_comunas = df_trans["comuna"].value_counts().nlargest(10).index
df_trans["comuna_agrupada"] = df_trans["comuna"].where(df_trans["comuna"].isin(top10_comunas), "OTROS")
frec_rut = df_trans["rut_razon_social"].value_counts()
df_trans["frecuencia_declaraciones_rut"] = df_trans["rut_razon_social"].map(frec_rut).astype(int)
 
# 4.5 Variables de Texto Libre: longitud en palabras de la glosa CIIU4
df_trans["ciiu4_n_palabras"] = df_trans["ciiu4"].astype(str).str.split().str.len()
 
# 4.6 Variables Nominales: One-Hot Encoding sobre estado_materia y comuna_agrupada
dummies_nom = pd.get_dummies(
    df_trans[["estado_materia", "comuna_agrupada"]],
    prefix=["estado", "comuna_top"],
    dtype=int,
)
df_trans = pd.concat([df_trans, dummies_nom], axis=1)
 
print(f"[OK] Transformaciones por rol completadas. Dimensiones actuales: {df_trans.shape[0]} filas x {df_trans.shape[1]} columnas")
display(df_trans[[
    "año", "region", "comuna_agrupada", "cantidad_kg", "es_gran_generador_bin",
    "n_peligrosidades", "multi_peligrosidad_bin", "estado_materia_ord", "frecuencia_declaraciones_rut"
]].head())


## 5. Escalamiento y Normalización: Comparación `StandardScaler` vs `MinMaxScaler` vs `RobustScaler`
Dado que `cantidad_kg` presenta un **16,19% de valores atípicos** y `latitud` un **15,37%** según el criterio $1,5 \times \text{RIC}$, se comparan los tres escaladores de `scikit-learn`:
- **`StandardScaler`** ($\frac{x - \mu}{\sigma}$): La desviación estándar de `cantidad_kg` ($\sigma = 438.010\text{ kg}$) está dominada por valores extremos, comprimiendo el rango intercuartílico del 50% central de los datos a apenas `0,0036`.
- **`MinMaxScaler`** ($\frac{x - \min}{\max - \min}$): El máximo de `48.223.740 kg` aplasta el 75% de las declaraciones de Chile contra el `0,0000` ($\text{RIC} = 0,0000$).
- **`RobustScaler`** ($\frac{x - Q_2}{\text{RIC}}$): Centra exactamente la mediana en **`0,0000`** y el rango intercuartílico en **`1,0000`**, por lo que se selecciona e incorpora al dataset procesado.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
 
COLS_ESCALAR = [
    "cantidad_kg", "latitud", "longitud",
    "frecuencia_declaraciones_rut", "n_peligrosidades", "ciiu4_n_palabras"
]
 
escaladores = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}
 
comparativa_escala = []
resultados_escala = {}
 
for nombre_esc, esc in escaladores.items():
    matriz = esc.fit_transform(df_trans[COLS_ESCALAR])
    df_esc = pd.DataFrame(matriz, columns=COLS_ESCALAR, index=df_trans.index)
    resultados_escala[nombre_esc] = df_esc
    for col in ["cantidad_kg", "latitud", "longitud"]:
        s = df_esc[col]
        comparativa_escala.append({
            "escalador": nombre_esc,
            "variable": col,
            "media": round(float(s.mean()), 4),
            "mediana": round(float(s.median()), 4),
            "ric": round(float(s.quantile(0.75) - s.quantile(0.25)), 4),
            "min": round(float(s.min()), 4),
            "max": round(float(s.max()), 4),
        })
 
df_comp_escala = pd.DataFrame(comparativa_escala)
display(df_comp_escala)
 
# Incorporar las columnas escaladas con RobustScaler (sufijo _rob)
df_rob = resultados_escala["RobustScaler"].add_suffix("_rob")
df_final = pd.concat([df_trans, df_rob], axis=1)
 
# Gráfico comparativo del efecto de los 3 escaladores sobre el percentil 0-95 de cantidad_kg
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (nombre_esc, df_esc) in zip(axes, resultados_escala.items()):
    s_recortada = df_esc["cantidad_kg"][df_esc["cantidad_kg"] <= df_esc["cantidad_kg"].quantile(0.95)]
    sns.histplot(s_recortada, bins=35, ax=ax, color="#0f766e")
    ax.set_title(f"{nombre_esc} — cantidad_kg (P0–P95)")
plt.tight_layout()
plt.show()


## 6. Validación Formal (`assert`), Pruebas Unitarias y Exportación de Productos de Fase 2
Se verifica programáticamente la integridad del dataset procesado y el contrato de las funciones del módulo `F2/src/preprocesamiento.py` con casos borde, exportando finalmente `F2/data/processed/base_maestra_2023_2025_clean.csv` y `F2/docs/bitacora_f2.md`.


In [ ]:
# 6.1 Verificaciones formales sobre el dataset procesado (df_final)
assert df_final.shape[0] == 118162, f"Se esperaban 118.162 filas únicas, hay {df_final.shape[0]}"
assert int(df_final.isna().sum().sum()) == 0, "No deben existir valores nulos en df_final"
assert int(df_final.duplicated().sum()) == 0, "No deben existir filas duplicadas en df_final"
assert df_final["region"].nunique() == 16, "La columna region debe tener exactamente las 16 regiones de Chile"
assert df_final["latitud"].between(-57.0, -17.0).all(), "Latitudes fuera del rango geográfico de Chile"
assert df_final["longitud"].between(-110.0, -65.0).all(), "Longitudes fuera del rango geográfico de Chile"
 
for col in COLS_ESCALAR:
    med_rob = float(df_final[f"{col}_rob"].median())
    assert abs(med_rob) < 1e-6, f"Mediana de {col}_rob debe ser 0.0, obtenido {med_rob}"
 
print("[OK] Todas las aserciones de integridad sobre df_final se cumplieron exitosamente.")
 
# 6.2 Pruebas unitarias de las funciones del módulo F2/src/preprocesamiento.py (incluyendo casos borde)
df_prueba = pd.DataFrame({
    "region": ["Araucanía", "La Araucanía", "Biobío", "Biobío"],
    "id_rubro": [10, 10, 20, 20],
    "id_peligrosidad": ["2,", "2,5,", "3,", "3,"],
    "peligrosidad": ["Tóxico crónico", "INFLAMABLE##TOXICO CRONICO", "Corrosivo", "CORROSIVO"],
    "estado_materia": ["sólido ", "LÍQUIDO", "sólido", "lodo"],
    "latitud": [-38735900.0, -38.7359, -36.8201, np.nan],
    "longitud": [-72590400.0, -72.5904, -73.0444, np.nan],
    "cantidad_kg": [100.0, np.nan, 500.0, 700.0],
    "id_lista_a": ["A3020", None, "A4070", "A4070"],
})
 
df_p1 = estandarizar_regiones_y_peligrosidad(df_prueba)
assert df_p1["region"].nunique() == 2, "Fallo en unificación de regiones"
assert df_p1.loc[1, "peligrosidad"] == "INFLAMABLE + TOXICO CRONICO", "Fallo en ordenamiento de peligrosidad"
 
df_p2 = rescatar_coordenadas_chile(df_p1)
assert abs(df_p2.loc[0, "latitud"] - (-38.7359)) < 1e-4, "Fallo en rescate de escala decimal de latitud"
 
df_p3 = imputar_por_grupo_con_flag(df_p2, ["cantidad_kg", "latitud", "longitud"], ["id_lista_a"])
assert int(df_p3.isna().sum().sum()) == 0, "Fallo en imputación de dataframe de prueba"
assert df_p3.loc[1, "cantidad_kg_imputado"] == 1, "Fallo en bandera binaria _imputado"
 
# Caso borde: cargar_datos_seguro debe lanzar FileNotFoundError ante ruta inexistente
try:
    cargar_datos_seguro("ruta_inexistente_12345.xlsx")
    raise AssertionError("Debió lanzar FileNotFoundError")
except FileNotFoundError:
    print("[OK] Prueba de caso borde superada: cargar_datos_seguro captura FileNotFoundError correctamente.")


In [ ]:
# 6.3 Exportación del dataset procesado y de la bitácora técnica de Fase 2
RUTA_CLEAN = Path("F2/data/processed/base_maestra_2023_2025_clean.csv")
RUTA_BITACORA = Path("F2/docs/bitacora_f2.md")
 
RUTA_CLEAN.parent.mkdir(parents=True, exist_ok=True)
RUTA_BITACORA.parent.mkdir(parents=True, exist_ok=True)
 
df_final.to_csv(RUTA_CLEAN, index=False, encoding="utf-8")
print(f"[OK] Dataset limpio exportado en: {RUTA_CLEAN} ({df_final.shape[0]} filas x {df_final.shape[1]} columnas)")
 
lineas_bitacora = [
    "# Bitácora Técnica de Decisiones — Fase 2 (Grupo 3 MCDI500)",
    "",
    f"- **Dataset de entrada (inmutable):** `{RUTA_RAW}` (`118.280 filas × 21 columnas`)",
    f"- **Dataset de salida (procesado):** `{RUTA_CLEAN}` (`{df_final.shape[0]} filas × {df_final.shape[1]} columnas`)",
    "",
    "## Registro de Decisiones Técnicas",
    "| Etapa | Problema detectado | Alternativas evaluadas | Decisión adoptada y justificación |",
    "| :--- | :--- | :--- | :--- |",
    "| 1. Deduplicación | 118 registros duplicados exactos en la base 2023–2025 | Conservar o eliminar duplicados exactos | Se eliminan con `drop_duplicates()`, quedando `118.162` declaraciones únicas. |",
    "| 2. Homologación regional y de peligrosidad | `region` tenía 19 etiquetas para 16 regiones y `peligrosidad` 812 variantes (`##` vs `;` en 2025) | Codificar en crudo vs normalizar diccionarios y cadenas | Se homologan las 16 regiones de Chile y se reducen las 812 variantes de `peligrosidad` a 93 categorías canónicas ordenadas. |",
    "| 3. Rescate geográfico 2024 | En 2024 las 39.518 coordenadas perdieron el punto decimal (ej. `-33447301481283`) | Descartar año 2024 vs rescatar punto decimal y filtrar rango de Chile | Se recupera la escala decimal validando `latitud` en `[-57, -17]` y `longitud` en `[-110, -65]`, rescatando más de 39.300 coordenadas. |",
    "| 4. Manejo de nulos | Nulos en `id_lista_a` (4,03%), coordenadas (0,24%) y `cantidad_kg` (0,02%) | `dropna()` (pierde 5.037 filas = 4,26%), media global o mediana/moda por grupo | Se imputa con mediana por `region` (numéricas) y moda por `id_rubro` (categóricas) más banderas `_imputado`, evitando sesgar `cantidad_kg` (media 16.422 kg vs mediana 234 kg). |",
    "| 5. Transformación por rol | Variables heterogéneas (nominales, ordinales, alta cardinalidad, fecha, texto) | Codificación única vs codificación específica por rol | Se crean binarias regulatorias (`es_gran_generador_bin >= 12.000 kg`), ordinal de estado físico, One-Hot, Top-10 comunas, frecuencia de RUT, armónicas de año y métricas de texto. |",
    "| 6. Escalamiento | `cantidad_kg` tiene 16,19% de outliers ($1,5 \\times \\text{RIC}$) | `StandardScaler`, `MinMaxScaler`, `RobustScaler` | Se selecciona `RobustScaler` (centra mediana en 0,0 y RIC en 1,0 sin aplastar el 75% de los datos como `MinMaxScaler`). |",
]
 
RUTA_BITACORA.write_text("\n".join(lineas_bitacora), encoding="utf-8")
print(f"[OK] Bitácora exportada en: {RUTA_BITACORA}")


In [ ]:
# 7. PRUEBAS UNITARIAS (NORMAL, LÍMITE Y EXCEPCIONES) Y RECURSIVIDAD (ID 2.1)

import F2.src.preprocesamiento as prep
SEMILLA = 42

# 7.1 Pruebas de funciones en caso normal, caso límite y captura de excepciones
print("=== 7.1 PRUEBAS DE FUNCIONES (CASO NORMAL, LÍMITE Y EXCEPCIONES) ===")
# a) Caso normal: normalización de peligrosidad y rescate de coordenada válida de 2024
assert prep._normalizar_glosa_peligrosidad("tóxico extrínseco; inflamable") == "INFLAMABLE + TOXICO EXTRINSECO"
assert prep._rescatar_valor_coordenada(-334473014, -57.0, -17.0) == -33.447301
print("[OK] Caso normal verificado: normalización léxica y rescate decimal correctos.")
# b) Caso límite: coordenada irreconciliable fuera de rango y valor nulo en glosa
assert np.isnan(prep._rescatar_valor_coordenada(999999, -57.0, -17.0))
assert prep._normalizar_glosa_peligrosidad(np.nan) == "SIN CLASIFICAR"
print("[OK] Caso límite verificado: valores extremos inválidos y NaN manejados sin romper el flujo.")
# c) Excepciones controladas (try/except): archivo inexistente y DataFrame vacío
excepciones_capturadas = 0
try:
    prep.cargar_datos_seguro("ruta_inexistente_prueba.xlsx")
except FileNotFoundError as err:
    excepciones_capturadas += 1
    print(f"[OK] Excepción 1 capturada (FileNotFoundError): {err}")
try:
    prep.diagnosticar_variables(pd.DataFrame())
except ValueError as err:
    excepciones_capturadas += 1
    print(f"[OK] Excepción 2 capturada (ValueError): {err}")
assert excepciones_capturadas == 2, "Ambas excepciones esperadas debieron capturarse."
# 7.2 Recursividad aplicada a metadatos jerárquicos del proyecto (Indicador ID 2.1)
print("\n=== 7.2 RECURSIVIDAD: APLANAMIENTO DE METADATOS ANIDADOS ===")
def aplanar_metadatos(diccionario: dict, prefijo: str = "") -> dict:
    """
    Aplana recursivamente un diccionario anidado de profundidad arbitraria
    en pares clave-valor separados por punto.
    """
    plano = {}
    for clave, valor in diccionario.items():
        nueva_clave = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict):
            plano.update(aplanar_metadatos(valor, nueva_clave))
        elif isinstance(valor, list):
            plano[nueva_clave] = ", ".join(str(x) for x in valor)
        else:
            plano[nueva_clave] = valor
    return plano
# Verificación de la función recursiva (casos plano, anidado profundo y vacío)
assert aplanar_metadatos({"a": 1}) == {"a": 1}, "Fallo en caso plano"
assert aplanar_metadatos({"a": {"b": {"c": 3}}}) == {"a.b.c": 3}, "Fallo en anidamiento profundo"
assert aplanar_metadatos({}) == {}, "Fallo en caso límite vacío"
print("[OK] Función recursiva 'aplanar_metadatos' superó todas las pruebas.")
METADATOS_F2 = {
    "proyecto": {
        "codigo": "MCDI500",
        "grupo": "Grupo 3",
        "fase": "F2",
        "semilla": SEMILLA,
    },
    "datos": {
        "entrada": {"archivo": "Base_Maestra_2023_2025.xlsx", "filas": int(df_raw.shape[0]), "columnas": int(df_raw.shape[1])},
        "salida": {"archivo": "base_maestra_2023_2025_clean.csv", "filas": int(df_final.shape[0]), "columnas": int(df_final.shape[1])},
    },
    "decisiones": {
        "imputacion": {"numericas": "mediana por region", "categoricas": "moda por id_rubro", "trazabilidad": "banderas _imputado"},
        "escalamiento": {"metodo_elegido": "RobustScaler", "motivo": "16.19% outliers IQR en cantidad_kg"},
    },
}
df_metadatos_planos = pd.DataFrame(
    [{"parametro": k, "valor": str(v)} for k, v in aplanar_metadatos(METADATOS_F2).items()]
)
display(df_metadatos_planos)

# 8. Reflexión Técnica (Cierre de Fase 2)

- **Coherencia entre las mediciones y las decisiones adoptadas:** La cuantificación previa de valores atípicos mediante el criterio $1,5 \times \text{RIC}$ demostró que `cantidad_kg` presenta un `16,19%` de valores extremos (`19.135` declaraciones) y una media (`16.437,52 kg`) 70 veces mayor que su mediana (`234,00 kg`). Esta evidencia objetiva guió de manera consistente tanto la **imputación por mediana regional** (que alteró la desviación estándar en apenas `-0,0080%`, frente a perder `5.037` filas con `dropna()`) como la elección de **`RobustScaler`** frente a `MinMaxScaler` (el cual comprimía el 75% de los datos en un `RIC = 0,0000`).
- **Aporte al problema de investigación:** La creación de los indicadores longitudinales por `id_vu` permitió responder preliminarmente la pregunta analizable: de los `12.192` establecimientos generadores registrados entre 2023 y 2025, un núcleo estable de **`6.785` establecimientos (`55,65%`)** declara en los tres años consecutivos y explica el **`83,41%` (`98.556` filas)** de todas las declaraciones de residuos peligrosos del país.